In [8]:
import os
import json
import itertools
import numpy as np
import pandas as pd
import vectorbt as vbt
import gc

# ==============================================================================
# 1. CARICAMENTO DELLE CONFIGURAZIONI JSON E MAPPING ASSET
# ==============================================================================
CONFIG_DIR = "progetto_trading_data"
FILE_CONFIG = {
    "3ema": "config_3ema.json",
    "macd_normale": "config_macd_normale.json",
    "macd_zerocross": "config_macd_zerocross.json",
    "donchian": "config_donchian.json"
}

configs = {}
all_tickers_sets = []

for key, filename in FILE_CONFIG.items():
    path = os.path.join(CONFIG_DIR, filename)
    if os.path.exists(path):
        with open(path, "r") as f:
            configs[key] = json.load(f)
        all_tickers_sets.append(set(configs[key]["ASSETS"].keys()))
    else:
        print(f"⚠️ Attenzione: File di configurazione {filename} non trovato.")

# Isoliamo i ticker comuni presenti in TUTTE le configurazioni caricate
if all_tickers_sets:
    ticker_comuni = sorted(list(set.intersection(*all_tickers_sets)))
    print(f"🎯 Rilevati {len(ticker_comuni)} ticker comuni su cui calcolare gli Ensemble Cross-Strat.")
else:
    raise FileNotFoundError("❌ Impossibile procedere: Nessun file di configurazione valido trovato.")

# ==============================================================================
# 2. CARICAMENTO DINAMICO DEI DATASET PARQUET REALI
# ==============================================================================
BASE_DIR = os.path.join("progetto_trading_data", "dataset_v3_parquet")
all_tickers_data = {}

if os.path.exists(BASE_DIR):
    for cluster in os.listdir(BASE_DIR):
        cluster_path = os.path.join(BASE_DIR, cluster)
        if os.path.isdir(cluster_path):
            for file in os.listdir(cluster_path):
                if file.endswith(".parquet"):
                    t = file.replace(".parquet", "")
                    if t in ticker_comuni:
                        all_tickers_data[t] = pd.read_parquet(os.path.join(cluster_path, file))
else:
    print(f"❌ ERRORE CRITICO: La cartella dei dataset Parquet {BASE_DIR} non esiste.")

# ==============================================================================
# 3. FUNZIONI HELPER PER METRICHE AVANZATE E GENERAZIONE SEGNALI NATIVI
# ==============================================================================

def calc_advanced_metrics(rets, giorni_anno):
    """Calcola le metriche di performance da una serie di rendimenti giornalieri"""
    if len(rets) == 0 or rets.isnull().all():
        return 0, 0, 0, 0, 0
    anni = len(rets) / giorni_anno
    cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    daily_std = rets.std()
    vol = daily_std * np.sqrt(giorni_anno)
    sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    neg_rets = rets[rets < 0]
    daily_down_std = neg_rets.std()
    sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    cum = (1 + rets).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    max_dd = dd.min() if len(dd) > 0 else 0
    return cagr, vol, sharpe, sortino, max_dd

# --- GENERATORI DI SEGNALI BASATI SUI VECCHI SCRIPT OPERATIVI ---

def get_3ema_signals(close, p):
    f, m, s = p[0], p[1], p[2]
    ema1 = vbt.MA.run(close, window=f, ewm=True).ma
    ema2 = vbt.MA.run(close, window=m, ewm=True).ma
    ema3 = vbt.MA.run(close, window=s, ewm=True).ma
    ent = (ema1.vbt.crossed_above(ema2) | ema1.vbt.crossed_above(ema3) | ema2.vbt.crossed_above(ema3)).vbt.signals.fshift(1)
    exi = (ema1.vbt.crossed_below(ema2) | ema1.vbt.crossed_below(ema3) | ema2.vbt.crossed_below(ema3)).vbt.signals.fshift(1)
    return ent, exi

def get_macd_normale_signals(close, p):
    f, s, sig = p[0], p[1], p[2]
    macd = vbt.MACD.run(close, fast_window=f, slow_window=s, signal_window=sig)
    ent = macd.macd.vbt.crossed_above(macd.signal).vbt.signals.fshift(1)
    exi = macd.macd.vbt.crossed_below(macd.signal).vbt.signals.fshift(1)
    return ent, exi

def get_macd_zerocross_signals(close, p):
    f, s, sig = p[0], p[1], p[2]
    macd = vbt.MACD.run(close, fast_window=f, slow_window=s, signal_window=sig)
    ent = macd.macd.vbt.crossed_above(0).vbt.signals.fshift(1)
    exi = macd.macd.vbt.crossed_below(macd.signal).vbt.signals.fshift(1)
    return ent, exi

def get_donchian_signals(close, high, low, p):
    u, l = p[0], p[1]
    hh = high.rolling(u).max().shift(1)
    ll = low.rolling(l).min().shift(1)
    ent = close.vbt.crossed_above(hh).vbt.signals.fshift(1)
    exi = close.vbt.crossed_below(ll).vbt.signals.fshift(1)
    return ent, exi

def genera_mappa_segnali(strat_name, close, high, low, lista_parametri):
    """Ritorna una lista di tuple (entries, exits) per ciascun set di parametri della strategia"""
    coppie_segnali = []
    for p in lista_parametri:
        if strat_name == "3ema":
            ent, exi = get_3ema_signals(close, p)
        elif strat_name == "macd_normale":
            ent, exi = get_macd_normale_signals(close, p)
        elif strat_name == "macd_zerocross":
            ent, exi = get_macd_zerocross_signals(close, p)
        elif strat_name == "donchian":
            ent, exi = get_donchian_signals(close, high, low, p)
        coppie_segnali.append((ent, exi))
    return coppie_segnali

# ==============================================================================
# 4. ENGINE DI COSTRUZIONE MATRICE ENSEMBLE CROSS-STRAT (LOGICA OR)
# ==============================================================================

def esegui_cross_ensemble(close, segnali_a, segnali_b, split_idx, giorni_anno):
    """Incrocia in modalità cartesiana i segnali con logica OR applicando split 60/40"""
    entries_slots, exits_slots = [], []
    
    # Costruiamo le colonne booleane della matrice tramite prodotto cartesiano
    for (ent_a, exi_a), (ent_b, exi_b) in itertools.product(segnali_a, segnali_b):
        entries_slots.append(ent_a | ent_b)
        exits_slots.append(exi_a | exi_b)
        
    entries_df = pd.concat(entries_slots, axis=1)
    exits_df = pd.concat(exits_slots, axis=1)
    
    # Suddivisione in compartimenti stagni IS e OOS
    close_is = close.iloc[:split_idx]
    close_oos = close.iloc[split_idx:]
    
    # Backtest parallelo sui singoli scompartimenti operativi della matrice
    pf_is = vbt.Portfolio.from_signals(close_is, entries_df.iloc[:split_idx], exits_df.iloc[:split_idx], freq='D')
    pf_oos = vbt.Portfolio.from_signals(close_oos, entries_df.iloc[split_idx:], exits_df.iloc[split_idx:], freq='D')
    
    # Conteggio e media dei trade eseguiti dalle sottocomponenti
    trd_is = pf_is.trades.count().mean()
    trd_oos = pf_oos.trades.count().mean()
    
    # Fusione dei rendimenti tramite media equiponderata (Equal-Weighted Blending delle fette di capitale)
    blended_rets_is = pf_is.returns().mean(axis=1)
    blended_rets_outs = pf_oos.returns().mean(axis=1)
    
    # 🔥 CORRETTO TYPO: Cambiato blended_is_rets con blended_rets_is
    cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(blended_rets_is, giorni_anno)
    cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(blended_rets_outs, giorni_anno)
    
    del entries_df, exits_df, pf_is, pf_oos
    gc.collect()
    
    return {
        "IS": {"Shp": shp_is, "Cagr": cagr_is, "DD": dd_is, "Sort": sort_is, "Trd": trd_is},
        "OOS": {"Shp": shp_oos, "Cagr": cagr_oos, "DD": dd_oos, "Sort": sort_oos, "Trd": trd_oos}
    }

# ==============================================================================
# 5. LOOP DI VALIDAZIONE GLOBALE E GENERAZIONE DASHBOARD TERMINALE
# ==============================================================================
# 🔥 ACCUMULATORE STRUTTURATO PER I WIDGET INTERATTIVI
opzioni_accumulate = []

for ticker in ticker_comuni:
    if ticker not in all_tickers_data:
        continue
        
    df = all_tickers_data[ticker]
    close = df['Close'].replace(0, np.nan).bfill().ffill().copy()
    high = df['High'].replace(0, np.nan).bfill().ffill().copy()
    low = df['Low'].replace(0, np.nan).bfill().ffill().copy()
    close.index = pd.to_datetime(close.index)
    
    # Estraiamo le specifiche di asset e calendario dal primo JSON utile
    meta_ref = configs["3ema"]["ASSETS"][ticker]
    giorni_anno = meta_ref["Giorni_Anno"]
    cluster = meta_ref["Cluster"]
    
    split_idx = int(len(close) * 0.6)
    close_is = close.iloc[:split_idx]
    close_oos = close.iloc[split_idx:]
    data_inizio_oos = meta_ref["Data_Inizio_OOS"]
    
    # ... Resto del codice identico fino alla generazione degli ensemble ...
    p_3ema = configs["3ema"]["ASSETS"][ticker]["Parametri_Ottimi"]
    p_macdn = configs["macd_normale"]["ASSETS"][ticker]["Parametri_Ottimi"]
    p_macdz = configs["macd_zerocross"]["ASSETS"][ticker]["Parametri_Ottimi"]
    p_donch = configs["donchian"]["ASSETS"][ticker]["Parametri_Ottimi"]
    
    s_3ema = genera_mappa_segnali("3ema", close, high, low, p_3ema)
    s_macdn = genera_mappa_segnali("macd_normale", close, high, low_range=None if 'low_range' not in locals() else low_range, lista_parametri=p_macdn) if False else genera_mappa_segnali("macd_normale", close, high, low, p_macdn)
    s_macdz = genera_mappa_segnali("macd_zerocross", close, high, low, p_macdz)
    s_donch = genera_mappa_segnali("donchian", close, high, low, p_donch)
    
    combo_3ema_macdn = esegui_cross_ensemble(close, s_3ema, s_macdn, split_idx, giorni_anno)
    combo_3ema_macdz = esegui_cross_ensemble(close, s_3ema, s_macdz, split_idx, giorni_anno)
    combo_macdn_donch = esegui_cross_ensemble(close, s_macdn, s_donch, split_idx, giorni_anno)
    combo_macdz_donch = esegui_cross_ensemble(close, s_macdz, s_donch, split_idx, giorni_anno)
    combo_3ema_donch  = esegui_cross_ensemble(close, s_3ema, s_donch, split_idx, giorni_anno)
    
    print("\n" + "="*110)
    print(f"📊 REPORT CROSS-STRATEGY ENSEMBLE PER: {ticker:<10} | Cluster: {cluster:<12} | Inizio OOS: {data_inizio_oos}")
    print("="*110)
    
    print("👑 PERFORMANCE STRATEGIE SINGOLE (Baseline JSON):")
    for s_id, label in [("3ema", "3EMA (Ensemble)    "), ("macd_normale", "MACD Normale (Top2)"), 
                        ("macd_zerocross", "MACD Zero-Cross(T2)"), ("donchian", "Donchian 2D (Top1) ")]:
        m_is = configs[s_id]["ASSETS"][ticker]["Metrics_In_Sample"]
        m_oos = configs[s_id]["ASSETS"][ticker]["Metrics_Out_Of_Sample"]
        
        p_str = ", ".join([str(tuple(x)) if len(x)==3 else str(tuple(x)) for x in configs[s_id]['ASSETS'][ticker]['Parametri_Ottimi']])
        
        print(f"   • [{label}] Params: [{p_str}]\n"
              f"     [IS]  Shp: {m_is['Sharpe_IS']:.2f} | Cagr: {m_is['CAGR_IS']*100:.1f}% | DD: {m_is['MaxDD_IS']*100:.1f}% | Trd: {m_is['Trades_IS']:.1f}\n"
              f"     [OOS] Shp: {m_oos['Sharpe_OOS']:.2f} | Cagr: {m_oos['CAGR_OOS']*100:.1f}% | DD: {m_oos['MaxDD_OOS']*100:.1f}% | Trd: {m_oos['Trades_OOS']:.1f}")
              
        # 🔥 MODIFICA APPLICATA: Salvataggio esteso di tutte le metriche storiche delle singole
        opzioni_accumulate.append({
            "Ticker": ticker, "Cluster": cluster, "Strategia": label.strip(),
            "SHP_IS": m_is['Sharpe_IS'], "CAGR_IS": m_is['CAGR_IS'], "DD_IS": m_is['MaxDD_IS'],
            "SHP_OOS": m_oos['Sharpe_OOS'], "CAGR_OOS": m_oos['CAGR_OOS'], "DD_OOS": m_oos['MaxDD_OOS'],
            "Parametri": f"[{p_str}]", "Parametri_Raw": {"tipo_file": s_id, "params": configs[s_id]['ASSETS'][ticker]['Parametri_Ottimi']}
        })
        
    print("-"*110)
    
    print("⚖️ BENCHMARK BUY & HOLD (Mercato Sottostante):")
    b_is = configs["3ema"]["ASSETS"][ticker]["Benchmark_In_Sample"]
    b_oos = configs["3ema"]["ASSETS"][ticker]["Benchmark_Out_Of_Sample"]
    print(f"   [IS]  B&H -> Shp: {b_is['Sharpe_BH_IS']:.2f} | Cagr: {b_is['CAGR_BH_IS']*100:.1f}% | DD: {b_is['MaxDD_BH_IS']*100:.1f}%\n"
          f"   [OOS] B&H -> Shp: {b_oos['Sharpe_BH_OOS']:.2f} | Cagr: {b_oos['CAGR_BH_OOS']*100:.1f}% | DD: {b_oos['MaxDD_BH_OOS']*100:.1f}%")
    print("-"*110)
    
    print("🔥 PERFORMANCE COMBINAZIONI HYBRID ENSEMBLE (Logica OR a Frazioni di Capitale Eque):")
    combos_data = [
        ("COMBO 1: [3EMA + MACD Normale]", combo_3ema_macdn, {"tipo_file": "hybrid_3ema_macdn", "params_a": p_3ema, "params_b": p_macdn}),
        ("COMBO 2: [3EMA + MACD Zero-Cross]", combo_3ema_macdz, {"tipo_file": "hybrid_3ema_macdz", "params_a": p_3ema, "params_b": p_macdz}),
        ("COMBO 3: [MACD Normale + Donchian]", combo_macdn_donch, {"tipo_file": "hybrid_macdn_donch", "params_a": p_macdn, "params_b": p_donch}),
        ("COMBO 4: [MACD Zero-Cross + Donchian]", combo_macdz_donch, {"tipo_file": "hybrid_macdz_donch", "params_a": p_macdz, "params_b": p_donch}),
        ("COMBO 5: [3EMA + Donchian]", combo_3ema_donch, {"tipo_file": "hybrid_3ema_donch", "params_a": p_3ema, "params_b": p_donch})
    ]
    
    for title, data, raw_info in combos_data:
        print(f"   👉 {title}\n"
              f"      [IS]  Ensemble -> Shp: {data['IS']['Shp']:.2f} | Cagr: {data['IS']['Cagr']*100:.1f}% | DD: {data['IS']['DD']*100:.1f}% | AvgTrd: {data['IS']['Trd']:.1f}\n"
              f"      [OOS] Ensemble -> Shp: {data['OOS']['Shp']:.2f} | Cagr: {data['OOS']['Cagr']*100:.1f}% | DD: {data['OOS']['DD']*100:.1f}% | AvgTrd: {data['OOS']['Trd']:.1f}")
              
        # 🔥 MODIFICA APPLICATA: Salvataggio esteso di tutte le metriche calcolate per gli ensemble
        opzioni_accumulate.append({
            "Ticker": ticker, "Cluster": cluster, "Strategia": title.split(" (")[0], 
            "SHP_IS": data["IS"]["Shp"], "CAGR_IS": data["IS"]["Cagr"], "DD_IS": data["IS"]["DD"],
            "SHP_OOS": data["OOS"]["Shp"], "CAGR_OOS": data["OOS"]["Cagr"], "DD_OOS": data["OOS"]["DD"],
            "Parametri": "Matrix Cross-Strat OR", "Parametri_Raw": raw_info
        })
        
    print("="*110)
    del s_3ema, s_macdn, s_macdz, s_donch
    del combo_3ema_macdn, combo_3ema_macdz, combo_macdn_donch, combo_macdz_donch, combo_3ema_donch
    gc.collect()

df_master_opzioni = pd.DataFrame(opzioni_accumulate)
print(f"\n📊 Tabellone Master generato in RAM con successo! {len(df_master_opzioni)} combinazioni pronte per l'interfaccia di spunta.")

🎯 Rilevati 90 ticker comuni su cui calcolare gli Ensemble Cross-Strat.

📊 REPORT CROSS-STRATEGY ENSEMBLE PER: AAPL       | Cluster: equities_cfd | Inizio OOS: 2023-06-20
👑 PERFORMANCE STRATEGIE SINGOLE (Baseline JSON):
   • [3EMA (Ensemble)    ] Params: [(6, 20, 245), (4, 20, 210), (4, 24, 145)]
     [IS]  Shp: 1.55 | Cagr: 36.6% | DD: -22.6% | Trd: 22.0
     [OOS] Shp: 0.42 | Cagr: 5.6% | DD: -23.8% | Trd: 21.3
   • [MACD Normale (Top2)] Params: [(4, 72, 80), (4, 64, 90)]
     [IS]  Shp: 1.70 | Cagr: 35.3% | DD: -17.8% | Trd: 13.5
     [OOS] Shp: 1.01 | Cagr: 15.0% | DD: -19.4% | Trd: 10.5
   • [MACD Zero-Cross(T2)] Params: [(4, 80, 80), (14, 36, 95)]
     [IS]  Shp: 1.32 | Cagr: 28.9% | DD: -23.0% | Trd: 10.5
     [OOS] Shp: 0.85 | Cagr: 13.8% | DD: -18.3% | Trd: 7.0
   • [Donchian 2D (Top1) ] Params: [(15, 5)]
     [IS]  Shp: 1.45 | Cagr: 28.1% | DD: -14.8% | Trd: 25.0
     [OOS] Shp: -0.02 | Cagr: -1.2% | DD: -19.2% | Trd: 18.0
------------------------------------------------------

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Questo dizionario conterrà l'albero finale delle tue scelte approvate
scelte_utente = {} 

# 1. Creazione dei componenti grafici interattivi
ticker_dropdown = widgets.Dropdown(options=ticker_comuni, description='Ticker:')
output_area = widgets.Output()
save_button = widgets.Button(description='Salva Selezione 💾', button_style='success')

def mostra_opzioni_ticker(change=None):
    """Funzione che si attiva ogni volta che si cambia ticker nel menu a tendina"""
    ticker_scelto = ticker_dropdown.value
    meta_ref = configs["3ema"]["ASSETS"][ticker_scelto]
    
    with output_area:
        clear_output(wait=True) # Pulisce l'area prima di ridisegnare i widget
        print(f"🎯 Asset: {ticker_scelto} | Cluster: {meta_ref['Cluster']} | Inizio OOS: {meta_ref['Data_Inizio_OOS']}\n")
        
        # Estrarre i record nativi del Buy & Hold dal JSON di base
        b_is = meta_ref.get("Benchmark_In_Sample", {})
        b_oos = meta_ref.get("Benchmark_Out_Of_Sample", {})
        print(f"⚖️ BENCHMARK BUY & HOLD (Mercato Sottostante):")
        print(f"   [IS]  B&H -> Sharpe: {b_is.get('Sharpe_BH_IS', 0):5.2f} | CAGR: {b_is.get('CAGR_BH_IS', 0)*100:5.1f}% | MaxDD: {b_is.get('MaxDD_BH_IS', 0)*100:5.1f}%")
        print(f"   [OOS] B&H -> Sharpe: {b_oos.get('Sharpe_BH_OOS', 0):5.2f} | CAGR: {b_oos.get('CAGR_BH_OOS', 0)*100:5.1f}% | MaxDD: {b_oos.get('MaxDD_BH_OOS', 0)*100:5.1f}%")
        print("-" * 145)
        
        print(f"Spunta le strategie che vuoi approvare. Lascia VUOTO e clicca Salva per SCARTARE il ticker:")
        print("-" * 145)
        
        # Filtriamo il DataFrame master accumulato nella cella precedente
        df_ticker = df_master_opzioni[df_master_opzioni['Ticker'] == ticker_scelto]
        
        ticker_checkboxes = []
        for idx, row in df_ticker.iterrows():
            # 🔥 LAYOUT AGGIORNATO: Specchio metrico completo e allineato al millimetro con il Buy & Hold
            label = (
                f"• {row['Strategia']:<36} | "
                f"[IS]  Strat -> Sharpe: {row['SHP_IS']:5.2f} | CAGR: {row['CAGR_IS']*100:5.1f}% | MaxDD: {row['DD_IS']*100:5.1f}% || "
                f"[OOS] Strat -> Sharpe: {row['SHP_OOS']:5.2f} | CAGR: {row['CAGR_OOS']*100:5.1f}% | MaxDD: {row['DD_OOS']*100:5.1f}% | "
                f"Params: {row['Parametri']}"
            )
            
            valore_precedente = scelte_utente.get(ticker_scelto, {}).get(row['Strategia'], False)
            checkbox_value = isinstance(valore_precedente, dict)
            
            cb = widgets.Checkbox(value=checkbox_value, description=label, layout=widgets.Layout(width='100%'))
            cb.metadata = row  # Agganciamo l'intera riga al widget per recuperarla al click del tasto salva
            display(cb)
            ticker_checkboxes.append(cb)
            
        # Passiamo la lista dei widget attivi al bottone di salvataggio
        save_button.current_checkboxes = ticker_checkboxes

def on_save_clicked(b):
    """Funzione che si attiva al click del bottone Salva"""
    ticker_scelto = ticker_dropdown.value
    checkboxes_correnti = getattr(save_button, 'current_checkboxes', [])
    
    # Resettiamo il cassetto di questo ticker per registrare fedelmente lo stato corrente delle spunte
    scelte_utente[ticker_scelto] = {}
        
    # Registriamo i checkbox attivi inserendo l'intero spettro metrico approvato
    for cb in checkboxes_correnti:
        strat_name = cb.metadata['Strategia']
        if cb.value: 
            scelte_utente[ticker_scelto][strat_name] = {
                "Dati_Configurazione": cb.metadata['Parametri_Raw'],
                "Cluster": cb.metadata['Cluster'],
                "Sharpe_IS_Approvato": float(cb.metadata['SHP_IS']),
                "Sharpe_OOS_Approvato": float(cb.metadata['SHP_OOS']),
                "CAGR_OOS_Approvato": float(cb.metadata['CAGR_OOS']),
                "MaxDD_OOS_Approvato": float(cb.metadata['DD_OOS'])
            }
            
    with output_area:
        # Controllo visivo immediato di approvazione o scarto esplicito
        if scelte_utente[ticker_scelto]:
            print(f"\n✅ Selezione per {ticker_scelto} salvata con successo in memoria temporanea!")
        else:
            print(f"\n❌ Nessuna strategia spuntata. {ticker_scelto} contrassegnato come SCARTATO dal portfolio!")

# 2. Collegamento degli eventi nativi
ticker_dropdown.observe(mostra_opzioni_ticker, names='value')
save_button.on_click(on_save_clicked)

# 3. Renderizzazione del blocco grafico
display(widgets.VBox([ticker_dropdown, output_area, save_button]))

# Inizializziamo il primo disegno automatico all'apertura della cella
mostra_opzioni_ticker()

In [11]:
# ==============================================================================
# 6. ESPORTAZIONE DEL PORTFOLIO APPROVATO IN JSON DEFINITIVO + REPORT 3D TOTAL
# ==============================================================================

# 1. FILTRO: Eliminiamo i ticker rimasti vuoti ({}) nella sessione corrente
portfolio_pulito = {ticker: strategie for ticker, strategie in scelte_utente.items() if strategie}

# 2. 🛡️ SCUDO DI PROTEZIONE: Evita di sovrascrivere il JSON se non hai selezionato nulla in questa sessione
if not portfolio_pulito:
    print("⚠️ ATTENZIONE: Non hai selezionato o salvato nessuna strategia in questa sessione!")
    print("❌ L'esportazione è stata BLOCCATA per evitare di sovrascrivere il file precedente con un database vuoto.")
else:
    path_portfolio_finale = os.path.join("progetto_trading_data", "portfolio_approvato.json")

    # Assicuriamoci che la cartella di destinazione esista
    os.makedirs(os.path.dirname(path_portfolio_finale), exist_ok=True)

    # Scrittura su disco fisso del database filtrato (leggero e pulito per il money management)
    with open(path_portfolio_finale, "w") as f:
        json.dump(portfolio_pulito, f, indent=4)

    # 3. 🏆 TABELLONE COMPLETO DI CONFERMA: Radiografia totale speculare delle metriche
    print("\n" + "═"*120)
    print(f"📦 SUCCESS-EXPORT: Portfolio su misura scritto in: {path_portfolio_finale} 🚀")
    print(f"Hai promosso con successo {len(portfolio_pulito)} asset diversi. Ecco il riepilogo metriche approvato:")
    print("═"*120)
    
    for ticker, strats in portfolio_pulito.items():
        # Estraiamo i metadati di riferimento e il Buy & Hold dai config globali
        meta_ref = configs["3ema"]["ASSETS"][ticker]
        cluster = meta_ref["Cluster"]
        b_is = meta_ref.get("Benchmark_In_Sample", {})
        b_oos = meta_ref.get("Benchmark_Out_Of_Sample", {})
        
        print(f"🎯 TICKER: {ticker:<10} | Cluster: {cluster:<12} | Inizio OOS: {meta_ref['Data_Inizio_OOS']}")
        print(f"   ⚖️  BENCHMARK BUY & HOLD (Mercato Sottostante):")
        print(f"      [IS]  B&H -> Sharpe: {b_is.get('Sharpe_BH_IS', 0):5.2f} | CAGR: {b_is.get('CAGR_BH_IS', 0)*100:5.1f}% | MaxDD: {b_is.get('MaxDD_BH_IS', 0)*100:5.1f}%")
        print(f"      [OOS] B&H -> Sharpe: {b_oos.get('Sharpe_BH_OOS', 0):5.2f} | CAGR: {b_oos.get('CAGR_BH_OOS', 0)*100:5.1f}% | MaxDD: {b_oos.get('MaxDD_BH_OOS', 0)*100:5.1f}%")
        print(f"   🔥 STRATEGIE APPROVATE:")
        
        for strat_name, data in strats.items():
            # 🔥 CROSS-REFERENCE: Recuperiamo lo specchio completo delle metriche dal DataFrame master della RAM
            row_data = df_master_opzioni[(df_master_opzioni['Ticker'] == ticker) & (df_master_opzioni['Strategia'] == strat_name)].iloc[0]
            
            print(f"      ↳ {strat_name:<38}")
            print(f"        [IS]  Strat -> Sharpe: {row_data['SHP_IS']:5.2f} | CAGR: {row_data['CAGR_IS']*100:5.1f}% | MaxDD: {row_data['DD_IS']*100:5.1f}%")
            print(f"        [OOS] Strat -> Sharpe: {row_data['SHP_OOS']:5.2f} | CAGR: {row_data['CAGR_OOS']*100:5.1f}% | MaxDD: {row_data['DD_OOS']*100:5.1f}%")
        print("-" * 120)
        
    print("\n🔥 Cabina di regia pronta. Il file JSON contiene i puntatori parametrici necessari per l'allocazione globale!")


════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
📦 SUCCESS-EXPORT: Portfolio su misura scritto in: progetto_trading_data\portfolio_approvato.json 🚀
Hai promosso con successo 56 asset diversi. Ecco il riepilogo metriche approvato:
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
🎯 TICKER: AAPL       | Cluster: equities_cfd | Inizio OOS: 2023-06-20
   ⚖️  BENCHMARK BUY & HOLD (Mercato Sottostante):
      [IS]  B&H -> Sharpe:  1.23 | CAGR:  42.7% | MaxDD: -31.4%
      [OOS] B&H -> Sharpe:  0.73 | CAGR:  17.0% | MaxDD: -33.4%
   🔥 STRATEGIE APPROVATE:
      ↳ MACD Normale (Top2)                   
        [IS]  Strat -> Sharpe:  1.70 | CAGR:  35.3% | MaxDD: -17.8%
        [OOS] Strat -> Sharpe:  1.01 | CAGR:  15.0% | MaxDD: -19.4%
----------------------------------------------------------------------------------------------------------------------